# Concatenate and align · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 18. Concatenate and align

Combine rows or columns while respecting index labels.

Before you start: Understand DataFrame indexes and missing values.

Official reference: https://pandas.pydata.org/docs/user_guide/merging.html#concat

### M18C01. Stack row batches

concat stacks tables when axis=0, the default. ignore_index=True assigns new position labels. This changes the index, not the Student_ID values.

**Read:** What is the list inside concat?

Answer: It contains the two DataFrames to combine.

**Predict:** How many student rows will result have?

Answer: 12, the 6 rows from each batch.

**Explain:** Does ignore_index remove duplicate students?

Answer: No. It only creates a fresh integer index; duplicate data would still remain.



In [ ]:
# Predict before running: How many student rows will result have?
df = students()
first = df.iloc[:6]
second = df.iloc[6:]
result = pd.concat([first, second], ignore_index=True)
print(result[["Student_ID", "Name"]])

**Your turn:** Stack the second batch before the first, and reset the index in the combined result.

For the check, keep these result names: `result`.

**Hint:** Reverse the order of the two DataFrames passed to concat.

**Watch for this:** Build a list of batches and concatenate once instead of repeatedly growing a DataFrame in a loop.

In [ ]:
# Change this code to complete the task.
# Question: Stack the second batch before the first, and reset the index in the combined result.
df = students()
result = pd.concat([df.iloc[6:], df.iloc[:6]], ignore_index=True)
print(result[["Student_ID", "Name"]])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.iloc[0]["Student_ID"] == "S007"
assert result.index.tolist() == list(range(12))
print("Code checks passed. Review the written explanation as well.")

S007 is first because concat keeps the order of the input batches.

**Quick check:** What changes with ignore_index=True?

1. The Student_ID values
2. The output row index
3. The column names

Answer: 2. A fresh range index is assigned; data values stay unchanged.

**Explain what you learned:** Use concat for batches with the same meaning and schema.

### M18C02. Align columns by labels

concat with axis=1 places objects side by side. It matches index labels, even when their order differs. This is why an ID index is safer than assuming row positions match.

**Read:** Why are the two Series given names?

Answer: Their names become output column labels.

**Predict:** Which score will Amira receive?

Answer: 88, because both values use index label S001.

**Explain:** Why is assigning scores.to_numpy() risky here?

Answer: It discards labels, so the first score, 72, would be assigned by position to Amira.



In [ ]:
# Predict before running: Which score will Amira receive?
names = pd.Series(["Amira", "Bilal"], index=["S001", "S002"], name="Name")
scores = pd.Series([72, 88], index=["S002", "S001"], name="Score")
result = pd.concat([names, scores], axis=1)
print(result)

**Your turn:** Add score 91 for S003, then concatenate using join="inner" so only shared IDs remain.

For the check, keep these result names: `result`.

**Hint:** Construct a three-value scores Series and set join="inner".

**Watch for this:** Resetting indexes before concatenating unrelated tables can hide a mismatched-key problem.

In [ ]:
# Change this code to complete the task.
# Question: Add score 91 for S003, then concatenate using join="inner" so only shared IDs remain.
names = pd.Series(["Amira", "Bilal"], index=["S001", "S002"], name="Name")
scores = pd.Series([72, 88, 91], index=["S002", "S001", "S003"], name="Score")
result = pd.concat([names, scores], axis=1, join="inner")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert set(result.index) == {"S001", "S002"}
assert result.loc["S001", "Score"] == 88
print("Code checks passed. Review the written explanation as well.")

S003 is excluded because its label is absent from names.

**Quick check:** What does axis=1 concatenate side by side?

1. Columns aligned on row labels
2. Rows matched by physical position
3. Only numeric values

Answer: 1. Column concatenation aligns the row indexes.

**Explain what you learned:** Alignment follows labels, not visual row order.

### M18C03. Request a specific index order

reindex requests specific labels in a specific order. A requested label that does not exist produces missing values. It can also omit existing labels that were not requested.

**Read:** What controls the row order?

Answer: The label list supplied to reindex.

**Predict:** What will appear for S099?

Answer: Missing Name and Score values because the ID does not exist.

**Explain:** How is this different from loc with a missing label?

Answer: With a list of IDs, loc raises KeyError if any requested ID is absent. reindex can create rows with missing values for absent labels. Label slices have different endpoint rules.



In [ ]:
# Predict before running: What will appear for S099?
df = students().set_index("Student_ID")
result = df.reindex(["S003", "S001", "S099"])[["Name", "Score"]]
print(result)

**Your turn:** Reindex to S012, S002, S001 and print the Name and Score columns in that order.

For the check, keep these result names: `result`.

**Hint:** Supply a different list of labels to reindex.

**Watch for this:** Reindexing generally requires unique labels on the axis being reindexed.

In [ ]:
# Change this code to complete the task.
# Question: Reindex to S012, S002, S001 and print the Name and Score columns in that order.
df = students().set_index("Student_ID")
result = df.reindex(["S012", "S002", "S001"])[["Name", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.index.tolist() == ["S012", "S002", "S001"]
assert result.notna().all().all()
print("Code checks passed. Review the written explanation as well.")

All three labels exist, so this result has no missing values.

**Quick check:** What does an absent label create during reindex?

1. A KeyError in every case
2. A row of missing values
3. A copy of the preceding row

Answer: 2. reindex creates missing entries for labels not present in the original unique index.

**Explain what you learned:** Reindex is an explicit request for labels and order.

### M18C04. Fill from a second source

combine_first fills missing values from matching labels in another object. Existing non-missing values in the first source win. Agree which source has priority before applying this rule.

**Read:** Which source has priority?

Answer: primary, because combine_first is called on it.

**Predict:** Will S001 become 85?

Answer: No. Its primary score 88 is already present.

**Explain:** Why is this different from an unconditional overwrite?

Answer: Only missing primary entries are filled; existing values remain.



In [ ]:
# Predict before running: Will S001 become 85?
primary = pd.Series([88, pd.NA, 91], index=["S001", "S002", "S003"], dtype="Int64")
backup = pd.Series([85, 72, 90], index=["S001", "S002", "S003"], dtype="Int64")
result = primary.combine_first(backup)
print(result)

**Your turn:** Reverse the source priority by calling combine_first on backup. Explain the S001 result.

For the check, keep these result names: `result`.

**Hint:** Place backup before the dot.

**Watch for this:** An unmatched backup label can expand the index; inspect the result size.

In [ ]:
# Change this code to complete the task.
# Question: Reverse the source priority by calling combine_first on backup. Explain the S001 result.
primary = pd.Series([88, pd.NA, 91], index=["S001", "S002", "S003"], dtype="Int64")
backup = pd.Series([85, 72, 90], index=["S001", "S002", "S003"], dtype="Int64")
result = backup.combine_first(primary)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == [85, 72, 90]
print("Code checks passed. Review the written explanation as well.")

Backup now wins wherever it has a value; S001 is 85.

**Quick check:** Which primary values are replaced by combine_first?

1. Every value
2. Only missing values with an available backup
3. Only zero values

Answer: 2. Existing primary values are retained, including legitimate zero values.

**Explain what you learned:** Filling from another source is a policy as well as a method.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates